# 01 — Business Understanding & Problem Framing

> *"Before you start building a machine learning model, you need to understand the business objective clearly."*  
> — Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch* (2025), Chapter 2

## 1. Business Context

### The Churn Problem in Retail Banking

Customer churn — the event where a bank client closes their account and leaves — is one of the most costly problems in retail banking. Industry research consistently shows that:

- **Acquiring a new customer costs 5–7× more** than retaining an existing one.
- A **5% reduction in churn** can increase profitability by **25–95%** (Harvard Business Review).
- Churned customers rarely return: once lost, the lifetime value (LTV) is permanently forfeited.

Despite this, most banks react *reactively* — they notice a customer has left only after the fact. The goal of this project is to shift the bank from a **reactive** to a **proactive** retention strategy by predicting which customers are most likely to churn *before* they do, and intervening with targeted offers, personalised outreach, or loyalty programs.

### Who Uses This Model?

| Stakeholder | How they use the model output |
|---|---|
| Retention team | Prioritise daily call lists by churn score |
| Marketing | Trigger automated retention emails for top-k at-risk customers |
| Product managers | Identify which product combinations correlate with churn |
| Risk / compliance | Forecast balance outflows for liquidity planning |

## 2. Objective

### Framing the Goal Precisely

Following Géron's recommendation to frame the ML task precisely before touching any data, we define:

> **Build a ranking model that scores every active bank customer by their probability of churning within the next observation window, so that the retention team can focus their limited resources on the customers most likely to leave.**

Key clarifications:

- **We want a ranker, not just a binary classifier.** A model that outputs `churn / no-churn` labels wastes information. A probability score lets the business decide the intervention threshold dynamically (e.g., top-500 customers today, adjust budget next month).
- **The output is a churn probability** `P(Exited=1 | customer features)`, not a hard label.
- **Batch inference, not real-time.** Scores are computed nightly and loaded into the CRM system.

### What Success Looks Like

A model is considered useful if it can identify the customers most likely to churn in the **top-k** prioritised list, measured by metrics described in section 4 below.

## 3. Performance Metrics

### Why Standard Accuracy Is Not Enough

With ~20% churn rate, a model that always predicts "no churn" achieves 80% accuracy — useless for business purposes. We need metrics that align with the business objective of **ranking and prioritising** customers.

---

### 3.1 ROC AUC (Area Under the ROC Curve)

**What it measures:** The probability that a randomly chosen churner is ranked higher than a randomly chosen non-churner.

**Business interpretation:** An AUC of 0.85 means that if you randomly pick one customer who churned and one who did not, the model assigns a higher risk score to the churner 85% of the time.

**Range:** 0.5 (random) → 1.0 (perfect). We target ≥ 0.85.

---

### 3.2 Average Precision (AP / PR-AUC)

**What it measures:** The area under the Precision-Recall curve — summarises how precisely we identify churners across all possible thresholds.

**Business interpretation:** More meaningful than ROC AUC when classes are imbalanced (~20% positives). It penalises models that produce many false alarms, which waste the retention team's time.

**Range:** equal to the prevalence rate (0.20) for a random model → 1.0 for a perfect model.

---

### 3.3 Recall@k

**What it measures:** Of all true churners in the dataset, what fraction appear in the top-k highest-scored customers?

$$\text{Recall@k} = \frac{\text{true churners in top-k}}{\text{total churners}}$$

**Business interpretation:** If the retention team can call 500 customers per month, Recall@500 tells us what percentage of the month's churners we would have reached. Target: Recall@500 ≥ 0.40.

---

### 3.4 Lift@k

**What it measures:** How many times better than random the model is at identifying churners in the top-k.

$$\text{Lift@k} = \frac{\text{Precision@k}}{\text{baseline prevalence}}$$

**Business interpretation:** A Lift@500 of 3.0 means the top-500 scored customers contain 3× more churners than a random sample of 500. This directly translates to ROI on retention spend.

---

| Metric | Why we use it | Target |
|---|---|---|
| ROC AUC | Overall ranking quality, threshold-free | ≥ 0.85 |
| Average Precision | Quality under class imbalance | ≥ 0.60 |
| Recall@500 | Coverage of risk outreach budget | ≥ 0.40 |
| Lift@500 | ROI multiplier on retention budget | ≥ 2.5× |

## 4. ML Problem Framing

Following Géron's taxonomy (Chapter 1), this problem is:

| Dimension | Choice | Reason |
|---|---|---|
| **Learning type** | Supervised | We have labelled historical data (`Exited = 0/1`) |
| **Task type** | Binary classification | Two outcomes: churned or not |
| **Output used** | Probability score | We rank, not label |
| **Learning mode** | Batch | Scores refreshed nightly from a full snapshot |
| **Label source** | Historical account closure events | Reliable ground truth from core banking system |

The model is **not** deployed as a real-time decision engine. It is a **batch scoring pipeline** that produces a ranked list consumed by downstream CRM workflows.

## 5. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
# If running from the notebooks/ subdirectory, go up one level
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

## 6. Load Configuration & Project Metadata

In [ ]:
import json
from client_churn_prediction.config import load_config

config = load_config(PROJECT_ROOT / "configs" / "project.toml")

print("=" * 55)
print("PROJECT METADATA")
print("=" * 55)
project = config.get("project", {})
data_cfg = config.get("data", {})
modeling_cfg = config.get("modeling", {})

print(f"  Name            : {project.get('name')}")
print(f"  Problem family  : {project.get('family')}")
print(f"  Problem type    : {project.get('problem_type')}")
print(f"  Business goal   : {project.get('business_goal')}")
print(f"  Primary metrics : {project.get('primary_metric')}")
print()
print("DATA CONFIG")
print("-" * 55)
print(f"  Train file      : {data_cfg.get('train_file')}")
print(f"  Target column   : {data_cfg.get('target')}")
print(f"  Positive label  : {data_cfg.get('positive_label')}")
print(f"  ID columns      : {data_cfg.get('id_columns')}")
print()
print("MODELING CONFIG")
print("-" * 55)
print(f"  Classifier      : {modeling_cfg.get('classifier')}")
print(f"  Class weight    : {modeling_cfg.get('class_weight')}")
print(f"  Test size       : {modeling_cfg.get('test_size')}")
print(f"  Random state    : {modeling_cfg.get('random_state')}")
print(f"  Top-k thresholds: {config.get('evaluation', {}).get('top_k')}")

## 7. Quick Data Peek

Before any deep analysis, we take a first glance at the raw data — shape, column names, data types, and the first few rows. This is step one of Géron's *Look at the Big Picture* phase.

In [ ]:
import pandas as pd
import numpy as np

from client_churn_prediction.data import load_training_frame

df = load_training_frame(config)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
print("Column data types:")
print(df.dtypes.to_string())

In [ ]:
df.head(5)

In [ ]:
df.describe().T.round(2)

## 8. Target Distribution Analysis

Understanding the **class imbalance** before any modelling decision is essential. Géron notes in Chapter 3 that imbalanced datasets require adapted strategies — both in metric selection and in the training procedure itself.

In [ ]:
import matplotlib.pyplot as plt

target_col = "exited"  # normalized snake_case from 'Exited'

counts = df[target_col].value_counts().sort_index()
pct = df[target_col].value_counts(normalize=True).sort_index() * 100

print("Target distribution:")
print("-" * 35)
for label, count in counts.items():
    label_name = "Churned (1)" if label == 1 else "Retained (0)"
    print(f"  {label_name}: {count:,} ({pct[label]:.1f}%)")

imbalance_ratio = counts[0] / counts[1]
print()
print(f"  Imbalance ratio (majority:minority): {imbalance_ratio:.1f}:1")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(
    ["Retained (0)", "Churned (1)"],
    counts.values,
    color=["#4C8CBF", "#E84545"],
    edgecolor="white",
    width=0.5,
)

for bar, pct_val in zip(bars, pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 80,
        f"{pct_val:.1f}%",
        ha="center", va="bottom", fontsize=12, fontweight="bold",
    )

ax.set_title("Target Variable Distribution: Exited", fontsize=13, pad=12)
ax.set_ylabel("Number of Customers")
ax.set_ylim(0, counts.max() * 1.15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**Observations:**
- The dataset is **moderately imbalanced** (~20% churn rate).
- This is not extreme enough to require aggressive resampling techniques such as SMOTE, but it does mean:
  - We will use `class_weight='balanced'` in tree-based and linear models.
  - We will evaluate with **Average Precision** and **Recall@k**, not raw accuracy.

## 9. Assumptions & Constraints

| Category | Statement |
|---|---|
| **Data freshness** | The snapshot represents a single point-in-time view; no temporal drift is modelled in this version |
| **Label definition** | `Exited = 1` means the customer closed their account during the observation period |
| **No causal inference** | The model is a *ranker*, not a causal model; correlation ≠ causation |
| **Deployment scope** | Batch scoring only; no real-time API in scope for this notebook series |
| **Feature availability** | All features used for training are available at inference time via the CRM system |
| **Privacy** | `RowNumber`, `CustomerId`, and `Surname` are excluded from model training |
| **Feedback loop** | No corrective labelling loop is implemented in v1; model re-training uses new snapshots |

## 10. Solution Strategy Roadmap

Following the ML project checklist in Géron Chapter 2, the project is structured as follows:

```
01 — Business Understanding     ← YOU ARE HERE
     Define the problem, metrics, and constraints

02 — Exploratory Data Analysis
     Understand distributions, correlations, anomalies

03 — Feature Engineering
     Domain-driven transformations, encoding, scaling

04 — Modelling & Evaluation
     Baseline → candidate models → hyperparameter search
     Threshold tuning, Lift/Recall@k evaluation

05 — Deployment & Consumption
     Serialise model, build scoring API, monitor drift
```

Each notebook is self-contained and reproducible. All paths are resolved relative to `PROJECT_ROOT` via `configs/project.toml`.